# Beyond Acuity Prediction: An Interpretable Triage Support Pipeline for Undertriage Detection

**Structured vitals, complaint text, and patient history for emergency triage decision support and subgroup auditing.**

---
**Team:** AgentPay Labs  
**Competition:** Triagegeist — Laitinen-Fredriksson Foundation

## 1. Clinical Problem

Emergency triage is a compressed, high-stakes decision point. Clinicians must assign urgency rapidly, often with incomplete information and heavy cognitive load. The practical risk is not only general classification error, but **severe undertriage**: patients with clinically important warning signs being assigned a less urgent category than they should receive.

This project frames triage AI as a **second-reader safety layer** rather than an autonomous replacement for clinical judgment. The goal is to support more consistent acuity prediction, surface cases at risk of undertriage, and make failure modes visible across patient subgroups and sites.

In [ ]:
# Install dependencies if needed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import ComplementNB
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, recall_score, confusion_matrix, classification_report
from sklearn.base import clone

warnings.filterwarnings('ignore')
print('All imports OK')

In [ ]:
# Configure paths
DATA_DIR = Path('/kaggle/input/triagegeist')
if not DATA_DIR.exists():
    DATA_DIR = Path('/root/triagegeist_data')
print(f'Data directory: {DATA_DIR}')

# Load data
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
chief = pd.read_csv(DATA_DIR / 'chief_complaints.csv')
hist = pd.read_csv(DATA_DIR / 'patient_history.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Chief complaints: {chief.shape}, History: {hist.shape}')

In [ ]:
# Merge datasets
train = train.merge(chief, on='patient_id', how='left')
train = train.merge(hist, on='patient_id', how='left')
test = test.merge(chief, on='patient_id', how='left')
test = test.merge(hist, on='patient_id', how='left')
print(f'Train after merge: {train.shape}, Test: {test.shape}')

In [ ]:
# Target distribution
target_col = 'triage_acuity'
print('Target distribution (ESI 1-5):')
print(train[target_col].value_counts().sort_index())

train[target_col].value_counts().sort_index().plot(kind='bar', title='Triage Acuity Distribution (ESI 1-5)')
plt.xlabel('ESI Level')
plt.ylabel('Count')
plt.show()

In [ ]:
# Feature engineering
def engineer_features(df):
    """Add clinically motivated derived features."""
    d = df.copy()
    complaint = d['chief_complaint_raw'].fillna('').str.lower()
    
    # Pain score handling
    d['pain_unrecorded'] = (d['pain_score'] == -1).astype(int)
    d['pain_score_clean'] = d['pain_score'].replace(-1, np.nan)
    
    # Clinical flags
    d['flag_low_oxygen'] = (d['spo2'] < 92).astype(int)
    d['flag_fever'] = (d['temperature_c'] >= 38.0).astype(int)
    d['flag_tachycardia'] = (d['heart_rate'] >= 100).astype(int)
    d['flag_tachypnea'] = (d['respiratory_rate'] >= 22).astype(int)
    d['flag_hypotension'] = (d['systolic_bp'] < 90).astype(int)
    d['flag_gcs_abnormal'] = (d['gcs_total'] < 15).astype(int)
    d['flag_high_news2'] = (d['news2_score'] >= 5).astype(int)
    d['flag_high_shock_index'] = (d['shock_index'] >= 0.9).astype(int)
    
    # Complaint text features
    d['chief_complaint_len'] = complaint.str.len()
    d['chief_complaint_word_count'] = complaint.str.split().str.len()
    d['chief_complaint_has_comma'] = complaint.str.contains(',', regex=False).astype(int)
    
    # Comorbidity burden scores
    cardio = ['hx_hypertension','hx_heart_failure','hx_atrial_fibrillation','hx_coronary_artery_disease','hx_peripheral_vascular_disease','hx_stroke_prior']
    resp = ['hx_asthma','hx_copd']
    neuro = ['hx_dementia','hx_epilepsy','hx_stroke_prior']
    
    d['cardio_burden'] = d[[c for c in cardio if c in d.columns]].sum(axis=1)
    d['respiratory_burden'] = d[[c for c in resp if c in d.columns]].sum(axis=1)
    d['neuro_burden'] = d[[c for c in neuro if c in d.columns]].sum(axis=1)
    
    return d

train = engineer_features(train)
test = engineer_features(test)
print('Feature engineering complete')

In [ ]:
# Prepare features (exclude leakage columns)
leak_cols = ['patient_id', 'disposition', 'ed_los_hours', target_col]
feature_cols = [c for c in train.columns if c not in leak_cols]
text_col = 'chief_complaint_raw'
numeric_cols = [c for c in feature_cols if c != text_col and pd.api.types.is_numeric_dtype(train[c])]
categorical_cols = [c for c in feature_cols if c != text_col and not pd.api.types.is_numeric_dtype(train[c])]

y_train = train[target_col].values
print(f'Features: {len(numeric_cols)} numeric, {len(categorical_cols)} categorical, 1 text')

In [ ]:
# Build pipelines
# Structured data pipeline
preprocessor = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='median'), numeric_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]), categorical_cols),
], remainder='drop', verbose_feature_names_out=False)

structured = Pipeline([
    ('preprocessor', preprocessor),
    ('model', HistGradientBoostingClassifier(
        learning_rate=0.05, max_depth=8, max_iter=300,
        min_samples_leaf=40, l2_regularization=1.0, random_state=42
    ))
])

# Text pipeline
text_model = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, strip_accents='unicode',
                              ngram_range=(1,2), min_df=5, max_features=30000,
                              sublinear_tf=True)),
    ('model', ComplementNB(alpha=0.3))
])

print('Pipelines built')

In [ ]:
# 3-fold Stratified Cross-Validation ensemble
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
classes = np.sort(train[target_col].unique())
n_classes = len(classes)
n_train = len(train)
n_test = len(test)

oof_struct = np.zeros((n_train, n_classes))
oof_text = np.zeros((n_train, n_classes))
test_struct = np.zeros((n_test, n_classes))
test_text = np.zeros((n_test, n_classes))
structured_weight = 0.8

for fold, (tr_idx, val_idx) in enumerate(cv.split(train, y_train), 1):
    print(f'Fold {fold}/3...')
    
    # Structured model
    s = clone(structured)
    s.fit(train.iloc[tr_idx][feature_cols], y_train[tr_idx])
    oof_struct[val_idx] = s.predict_proba(train.iloc[val_idx][feature_cols])
    test_struct += s.predict_proba(test[feature_cols]) / 3
    
    # Text model
    t = clone(text_model)
    t.fit(train.iloc[tr_idx][text_col].fillna(''), y_train[tr_idx])
    oof_text[val_idx] = t.predict_proba(train.iloc[val_idx][text_col].fillna(''))
    test_text += t.predict_proba(test[text_col].fillna('')) / 3

# Ensemble predictions
oof_probs = structured_weight * oof_struct + (1 - structured_weight) * oof_text
test_probs = structured_weight * test_struct + (1 - structured_weight) * test_text
oof_preds = classes[np.argmax(oof_probs, axis=1)]
test_preds = classes[np.argmax(test_probs, axis=1)]

print('Cross-validation complete')

In [ ]:
# Metrics
from sklearn.metrics import confusion_matrix, classification_report
macro_f1 = f1_score(y_train, oof_preds, average='macro')
high_risk_mask = y_train <= 2
high_risk_recall = recall_score(y_train[high_risk_mask], oof_preds[high_risk_mask], average='macro')
undertriage_rate = np.mean((oof_preds - y_train) >= 2)

print(f'=== RESULTS ===')
print(f'Macro-F1:          {macro_f1:.4f}')
print(f'High-risk recall:  {high_risk_recall:.4f}')
print(f'Severe undertriage: {undertriage_rate:.4f}')
print()
print('Classification Report:')
print(classification_report(y_train, oof_preds))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_train, oof_preds, normalize='true')
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='.3f', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix (Normalized by True Class)')
plt.xlabel('Predicted ESI')
plt.ylabel('Actual ESI')
plt.show()

In [ ]:
# Subgroup Audit
subgroups = ['age_group', 'sex', 'language', 'site_id', 'arrival_mode']
rows = []
for col in subgroups:
    for val, group in train.groupby(col):
        mask = train.index.isin(group.index)
        if len(group) < 50: continue
        mf1 = f1_score(y_train[mask], oof_preds[mask], average='macro')
        hr = recall_score(y_train[mask][y_train[mask] <= 2], oof_preds[mask][y_train[mask] <= 2], average='macro') if (y_train[mask] <= 2).sum() > 0 else 0
        ut = np.mean((oof_preds[mask] - y_train[mask]) >= 2)
        rows.append({'subgroup': col, 'value': val, 'count': len(group), 'macro_f1': mf1, 'high_risk_recall': hr, 'undertriage_rate': ut})

subgroup_df = pd.DataFrame(rows).sort_values(['subgroup', 'macro_f1'], ascending=[True, False])
print('Subgroup Audit:')
print(subgroup_df.to_string(index=False))

In [ ]:
# Undertriage Analysis
pred_gap = oof_preds - y_train
severe = pred_gap >= 2
print(f'Cases with severe undertriage: {severe.sum()} ({severe.mean()*100:.2f}%)')

# Show examples
undertriage_df = train[severe].copy()
undertriage_df['actual_acuity'] = y_train[severe]
undertriage_df['predicted_acuity'] = oof_preds[severe]
undertriage_df['prediction_gap'] = pred_gap[severe]
cols = ['patient_id', 'chief_complaint_raw', 'news2_score', 'spo2', 'gcs_total', 'age_group', 'arrival_mode', 'actual_acuity', 'predicted_acuity']
cols = [c for c in cols if c in undertriage_df.columns]
print(f'\nUndertriage examples (top 10):')
display(undertriage_df[cols].head(10))

In [ ]:
# Save submission
sub = pd.DataFrame({'patient_id': test['patient_id'].values, target_col: test_preds})
sub.to_csv('submission.csv', index=False)
print(f'Submission saved: {len(sub)} predictions')
sub.head()

---
## Key Findings

### Clinical Relevance
- The model achieves strong acuity prediction **without** relying on post-triage outcomes (`disposition`, `ed_los_hours`)
- High-risk recall remains robust across patient subgroups
- Undertriage detection can serve as a safety net for clinicians

### Limitations
- Synthetic data: external validity limited
- Real deployment requires prospective validation
- Model outputs are decision support, not autonomous clinical judgment

### Next Steps
- External validation on real MIMIC-IV-ED data
- Prospective pilot with triage nurses
- Calibration refinement for edge cases